# Model Tester

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
import sys

os.environ["KERAS_BACKEND"] = "tensorflow"
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
# os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
# os.environ["XLA_FLAGS"] = (
#     "--xla_gpu_cuda_data_dir=/hpc/mp/apps/nvidia/hpc_sdk/24.5/Linux_x86_64/24.5/cuda"
# )

In [3]:
import logging
import time

import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from keras.optimizers import Adam
from keras.optimizers.schedules import ExponentialDecay, LearningRateSchedule
from keras.metrics import RootMeanSquaredError

from mlpng import Core

# from mlpng.models import AutoModel
# from mlpng.models.modelcore import get_model_class
from mlpng.utils import (
    setup_logging,
    load_data,
    get_fisher,
    plot_predictions,
    plot_histogram,
    print_errors,
    plot_metrics,
    WarmupLearningRate,
    AttentionSchedule,
    try_init_wandb,
)
from mlpng.utils.dataloaders import HDF5Dataset
from deepsphere import HealpyGCNN, healpy_layers as hp_layer
from deepsphere.healpy_layers import HealpyChebyshev, HealpyPool

2024-11-21 15:59:29.519168: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-21 15:59:29.519233: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-21 15:59:29.520326: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-21 15:59:29.527289: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-21 15:59:30.751513: W tensorflow/compiler/tf2

In [4]:
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")
print(f"keras version: {keras.__version__}")
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print("Python executable:", sys.executable)
print("Python version:", sys.version)

TensorFlow version: 2.15.1
CUDA version: 12.2
cuDNN version: 8
keras version: 2.15.0
Conda environment: deepsphere
Python executable: /work/users/stevensonb/.conda/envs/deepsphere/bin/python
Python version: 3.10.15 | packaged by conda-forge | (main, Oct 16 2024, 01:24:24) [GCC 13.3.0]


In [5]:
logger = setup_logging(__name__, level=logging.DEBUG)

In [6]:
MAX_EPOCHS = 30
BATCH_SIZE = 64

args = [
    "settings/n128.json",
    "--nsims",
    "100",
    "--narray",
    "1000",
    "--pols",
    "T",
    "--fnl_range",
    "-100",
    "100",
]

core = Core(args)
indices = np.arange(hp.nside2npix(core.nside))

# fisher = get_fisher(core.file)
# fnl_scale = (core.fnl_max - core.fnl_min) / 2.0
# scaled_std = 1 / np.sqrt(fisher) / fnl_scale

21-Nov-24 15:59:33 - mlpng.core - INFO - Parsing CLI args: ['settings/n128.json', '--nsims', '100', '--narray', '1000', '--pols', 'T', '--fnl_range', '-100', '100']
21-Nov-24 15:59:33 - mlpng.core - INFO - Loading settings from file 'settings/n128.json'
21-Nov-24 15:59:33 - mlpng.core - DEBUG - Forcing setting 'nsims' to 100 due to CLI
21-Nov-24 15:59:33 - mlpng.core - DEBUG - Forcing setting 'narray' to 1000 due to CLI
21-Nov-24 15:59:33 - mlpng.core - DEBUG - Forcing setting 'fnl_range' to [-100.0, 100.0] due to CLI
21-Nov-24 15:59:33 - mlpng.core - DEBUG - Forcing setting 'pols' to ['T'] due to CLI
21-Nov-24 15:59:33 - mlpng.core - DEBUG - Found non-default value for 'cosmo_params': {'H0': 67.66, 'As': 2.1056e-09, 'ns': 0.9665, 'ombh2': 0.02242, 'omch2': 0.11933, 'tau': 0.0561, 'pivot_scalar': 0.05} (default: {'As': 2.13e-09, 'ns': 0.9624, 'pivot_scalar': 0.05})
21-Nov-24 15:59:33 - mlpng.core - DEBUG - Overriding cosmo param 'As' from 2.13e-09 to 2.1056e-09
21-Nov-24 15:59:33 - mlp

In [7]:
def to_tf(ds):
    return (
        tf.data.Dataset.from_generator(
            lambda: ds,
            output_signature=(
                tf.TensorSpec(shape=(len(indices), 1), dtype=tf.float32),
                tf.TensorSpec(shape=(), dtype=tf.float32),
            ),
        )
        .apply(tf.data.experimental.assert_cardinality(len(ds)))
        .cache()
        .shuffle(buffer_size=1024, reshuffle_each_iteration=True)
        .batch(
            BATCH_SIZE,
            drop_remainder=True,
            num_parallel_calls=tf.data.AUTOTUNE,
            deterministic=False,
        )
        .prefetch(tf.data.AUTOTUNE)
    )


f = 10
ds = HDF5Dataset(core.file, x_name="alm", y_name="fnl")
train_ds, test_ds, val_ds = ds.split(0.7 / f, 0.2 / f, 0.1 / f, verbose=True)
train_tf, test_tf, val_tf = map(to_tf, (train_ds, test_ds, val_ds))

21-Nov-24 15:59:33 - mlpng.utils.dataloaders - INFO - Splitting data into train: 6999, val: 2000, test: 1000


2024-11-21 15:59:34.627334: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79099 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:87:00.0, compute capability: 8.0


In [ ]:
K = 11
depth = 1
cnn_depth = 2
F0 = 16

layers = []
for i in range(depth):
    for j in range(cnn_depth):
        layers.append(
            HealpyChebyshev(
                K=K,
                Fout=F0 * 2**i,
                use_bias=True,
                use_bn=False,
                activation="gelu" if j == 0 else None,
            )
        )
    layers.append(tf.keras.layers.LayerNormalization())
    layers.append(HealpyPool(p=1, pool_type="MAX"))

layers.append(keras.layers.Flatten())
layers.append(keras.layers.Dense(512, "gelu"))
layers.append(keras.layers.Dense(1))


decay_steps = len(train_ds) // BATCH_SIZE  # once per epoch
learning_rate = ExponentialDecay(1e-5, decay_steps, 0.95, staircase=True)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    indices = np.arange(hp.nside2npix(core.nside))
    gcnn = HealpyGCNN(
        nside=core.nside,
        indices=indices,
        layers=layers,
        n_neighbors=20,
        max_batch_size=BATCH_SIZE,
        initial_Fin=1,
    )
    gcnn.build(input_shape=(None, len(indices), 1))
    # gcnn.summary()

    # here is how to use the gcnn as a layer, KEEP this
    # inputs = keras.layers.Input(shape=(len(indices), 1))
    # layer = gcnn(inputs)
    # # layer = keras.layers.Flatten()(layer)
    # # layer = keras.layers.Dense(128, "relu")(layer)
    # # layer = keras.layers.Dense(32)(layer)
    # # layer = keras.layers.Dense(1)(layer)
    # model = keras.Model(inputs=inputs, outputs=layer)
    model = gcnn

    metrics = []  # RootMeanSquaredError(), "mae"]
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate), loss="mse", metrics=metrics
    )

model.summary()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
21-Nov-24 15:59:35 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
Detected a reduction factor of 2.0, the input with nside 128 will be transformed to 64 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul 

: 

In [9]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    TensorBoard(log_dir=f"data/tensorboard/notebooks/{time.strftime('%Y%m%d-%H%M%S')}"),
    TerminateOnNaN(),
    keras.callbacks.ModelCheckpoint(
        f"{core.dirs['base']}/models/deepsphere_1.keras",
        monitor="val_loss",
        save_best_only=True,
        initial_value_threshold=400,
    ),
]

try_init_wandb(notes="model testing", tags=["notebook"], append_to=callbacks)

history = model.fit(
    train_tf, epochs=MAX_EPOCHS, validation_data=val_tf, callbacks=callbacks
)

21-Nov-24 16:00:48 - mlpng.utils.utils - DEBUG - wandb version: 0.18.7


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jbrandons (mlpng). Use `wandb login --relogin` to force relogin


21-Nov-24 16:00:50 - tensorflow - WARNING - Model's `__init__()` arguments contain non-serializable objects. Please implement a `get_config()` method in the subclassed Model for proper saving and loading. Defaulting to empty config.
21-Nov-24 16:00:50 - tensorflow - WARNING - Model's `__init__()` arguments contain non-serializable objects. Please implement a `get_config()` method in the subclassed Model for proper saving and loading. Defaulting to empty config.


2024-11-21 16:00:50.080556: W tensorflow/core/grappler/optimizers/data/auto_shard.cc:553] The `assert_cardinality` transformation is currently not handled by the auto-shard rewrite and will be removed.


Epoch 1/30
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Tracing... Due to tensor size, tf.sparse.sparse_dense_matm

F0000 00:00:1732226458.673908 3543204 gpu_launch_config.h:129] Check failed: work_element_count >= 0 (-2080374784 vs. 0) 


: 

: 

In [ ]:
model.evaluate(test_tf, verbose=1)
preds = model.predict(test_tf, verbose=1)  # .flatten()
truth = np.concatenate([y for _, y in test_tf])

In [ ]:
fisher = get_fisher(core.file)
print_errors(truth, preds, fisher)

save_base = core.dirs["plot"]
plot_metrics(history, metrics=["loss"], save_file=save_base + "/ds-loss.png")
plot_predictions(
    truth, preds, fisher=fisher, show=True, save_file=save_base + "/ds-preds.png"
)
plot_histogram(truth, preds, show=True, save_file=save_base + "/ds-hist.png")